In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import *
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
spark = (
    SparkSession.builder
      .config("spark.driver.memory", "48g")                 # ajuste p/ 16g–24g
      .config("spark.sql.execution.arrow.pyspark.enabled", "true")
      .config("spark.sql.execution.arrow.maxRecordsPerBatch", "20000")
      .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)  # 64MB/partição
      .config("spark.driver.maxResultSize", "0")            # sem limite de resultado (cuidado)
      .getOrCreate()
)

spark.version

'4.0.1'

In [3]:
# read table
data = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/dados/internacao.parquet"
)
data.count()

334560

In [4]:
# filter pacientes in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/target_internacao.parquet"
).select("prontuario").distinct()

data = data.join(pacientes_target, on="prontuario", how="inner")
data.count()

235748

In [5]:
# converte as strings para timestamp
data = data.withColumn("dthr_internacao", F.to_timestamp("dthr_internacao")) \
       .withColumn("dthr_alta_medica", F.to_timestamp("dthr_alta_medica"))

# calcula a diferenca em dias (alta - internação)
data = data.withColumn("dias_internado",
                   F.round(F.datediff("dthr_alta_medica", "dthr_internacao"), 2))

In [6]:
# filter cases above august 2022
data = data.filter(F.col("dthr_alta_medica") >= F.lit("2022-08-01"))

In [7]:
# calcula a diferenca em dias (alta - internação)
data = data.withColumn("dias_internado",
                   F.round(F.datediff("dthr_alta_medica", "dthr_internacao"), 2))

# cid agg
data = data.withColumn("cid_agg", F.substring(F.col("cid"), 1, 1))

In [8]:
data.show(10)

+-----------+-----------+-----+--------------------+-------------------+-------------------+--------------------+---------------------------------+-----------------------+---------------------+--------+--------------------------+--------------+-------+
| prontuario|atendimento|  cid|     dthr_internacao|   dthr_alta_medica|  dt_saida_paciente|  descricao_anamnese|descricao_nota_adicional_anamnese|descricao_origem_evento|flag_obito_internacao|dt_obito|descricao_tipo_alta_medica|dias_internado|cid_agg|
+-----------+-----------+-----+--------------------+-------------------+-------------------+--------------------+---------------------------------+-----------------------+---------------------+--------+--------------------------+--------------+-------+
|2.1231147E7|     358884|G93.4|2021-09-27 06:39:...|2022-09-08 14:38:00|2022-09-08 20:56:00|Identificação pac...|             MICHELLE DUARTE D...|             EMERGÊNCIA|                    N|    NULL|      ALTA COM PREVISÃO...|           3

In [9]:
# ---- parâmetros ----
min_support = 1000        # mínimo de registros por CID para entrar no ranking (ajuste)
high_pct = 0.80          # corte para "alta permanência" (P80 por padrão)

# ---- sanity: tipagem e nulos ----
data1 = (
    data.select("cid_agg" , F.col("dias_internado").cast("double").alias("dias_internado"))
      .filter(F.col("cid_agg").isNotNull() & F.col("dias_internado").isNotNull())
)

In [10]:
# ---- 1) define "alta permanência" pelo percentil global (ex.: P80) ----
p80 = data1.select(F.expr(f"percentile_approx(dias_internado, {high_pct})").alias("p")).collect()[0]["p"]
data_flag = data1.withColumn("is_high_stay", (F.col("dias_internado") >= F.lit(p80)).cast("int"))
p80

13.0

In [11]:
# ---- 2) taxa global de alta permanência ----
p_global = data_flag.agg(F.avg("is_high_stay").alias("p")).collect()[0]["p"]
p_global

0.2110150862167461

In [12]:
# ---- 3) métricas por CID: suporte, taxa de alta, lift, médias úteis ----
by_cid = (
    data_flag.groupBy("cid_agg")
           .agg(
               F.count("*").alias("n"),
               F.avg("is_high_stay").alias("rate_high"),
               F.avg("dias_internado").alias("mean_days"),
               F.expr("percentile_approx(dias_internado, 0.5)").alias("p50_days"),
               F.expr("percentile_approx(dias_internado, 0.9)").alias("p90_days"),
           )
           .withColumn("lift_high", F.col("rate_high") / F.lit(p_global))
)
display(by_cid)

DataFrame[cid_agg: string, n: bigint, rate_high: double, mean_days: double, p50_days: double, p90_days: double, lift_high: double]

In [13]:

# ---- 4) filtra por suporte e pega Top-50 por lift (ou troque para mean_days se preferir) ----
top50_data = (
    by_cid.filter(F.col("n") >= F.lit(min_support))
          .orderBy(F.col("lift_high").desc(), F.col("p90_days").desc())
          .limit(50)
          .select("cid_agg")
)

top50 = [r[0] for r in top50_data.collect()]

In [14]:

# ---- 5) coluna final com "Outros" para quem não está no Top-50 ----
data = data.withColumn(
    "cid_agg_top50",
    F.when(F.col("cid_agg" ).isin(top50), F.col("cid_agg" )).otherwise(F.lit("Outros"))
)

In [15]:
def generate_inter_features(data):

    # groupby average value by prontuario + cid_agg_top50
    w_prompt_param = W.partitionBy("prontuario", "cid_agg_top50")

    agg_mean =( 
               (data.withColumn("qtd_media_dias_internado", F.mean("dias_internado").over(w_prompt_param))
                .select(
                    "prontuario",
                    "cid_agg_top50",
                    "qtd_media_dias_internado"
                    )
        ).groupBy("prontuario")
        .pivot("cid_agg_top50")
        .agg(F.first("qtd_media_dias_internado"))
    )

    
    # renomeamos as colunas para adicionar os sufixos
    for col in agg_mean.columns[1:]:
        agg_mean = agg_mean.withColumnRenamed(col, f"{col.lower()}_qtd_media_dias_internado")

    # groupby max date value by prontuario + param_final
    w_orderdate = W.partitionBy("prontuario", "cid_agg_top50").orderBy(F.col("dthr_alta_medica").desc())

    agg_max_date = (
                (data.withColumn("rn", F.row_number().over(w_orderdate))
                .filter(F.col("rn") == 1)
                .select(
                   "prontuario",
                    "cid_agg_top50",
                    "dias_internado"
                )
        ).groupBy("prontuario")
        .pivot("cid_agg_top50")
        .agg(F.first("dias_internado"))
    )

    for col in agg_max_date.columns[1:]:
        agg_max_date = agg_max_date.withColumnRenamed(col, f"{col.lower()}_valor_max_date")


    # join the two datasets
    features = agg_mean.join(agg_max_date, on="prontuario", how="inner")

    return features

In [16]:
def generate_qtd_window_features(data, ref_ts):

    # primeiro periodo
    data_filtered1 = data.filter(
    F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -3)
    )
    agg1 = data_filtered1.select("prontuario").groupBy("prontuario").count()
    agg1 = agg1.withColumnRenamed("count", "qtd_internacao_3_0_m")

    # segundo periodo
    data_filtered2 = data.filter(
    (    F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -6)) &
    (F.col("dthr_alta_medica") <= F.add_months(F.lit(ref_ts), -3))
    )
    agg2 = data_filtered2.select("prontuario").groupBy("prontuario").count()
    agg2 = agg2.withColumnRenamed("count", "qtd_internacao_6_3_m")

    # terceiro periodo
    data_filtered3 = data.filter(
    (    F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -9)) &
    (F.col("dthr_alta_medica") <= F.add_months(F.lit(ref_ts), -6))
    )
    agg3 = data_filtered3.select("prontuario").groupBy("prontuario").count()
    agg3 = agg3.withColumnRenamed("count", "qtd_internacao_9_6_m")

    # quarto periodo
    data_filtered4 = data.filter(
    (    F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -12)) &
    (F.col("dthr_alta_medica") <= F.add_months(F.lit(ref_ts), -9))
    )
    agg4 = data_filtered4.select("prontuario").groupBy("prontuario").count()
    agg4 = agg4.withColumnRenamed("count", "qtd_internacao_12_9_m")

    # join all features
    features = agg1.join(agg2, on="prontuario", how="outer") \
                   .join(agg3, on="prontuario", how="outer") \
                   .join(agg4, on="prontuario", how="outer")
    
    features = features.fillna(0)

    return features

In [17]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("dthr_alta_medica")).alias("min_m"),
    F.date_trunc("month", F.max("dthr_alta_medica")).alias("max_m"),
)

ref_dates_data = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_data.collect()]
len(ref_dates)

38

In [18]:
# define dataframe para incorporar dados ao cursor
features = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data.filter(
        (F.col("dthr_alta_medica") < F.lit(ref_ts)) &
        (F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -24))
    )
    
    features_dias_internado = generate_inter_features(data_filtered)

    features_qtd_internacoes = generate_qtd_window_features(data_filtered, ref_ts)


    features_internacao_refdate = features_dias_internado.join(features_qtd_internacoes, on="prontuario", how="outer")

    features_internacao_refdate = features_internacao_refdate.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    features = features.unionByName(features_internacao_refdate, allowMissingColumns=True)

    print(ref_ts)

2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00
2025-04-01 00:00:00
2025-05-01 00:00:00
2025-06-01 00:00:00
2025-07-01 00:00:00
2025-08-01 00:00:00
2025-09-01 00:00:00


In [19]:
# filter pacientes e date_ref in target data
pacientes_target = spark.read.option("header", True).parquet(
    "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/target_internacao.parquet"
).select("prontuario", "date_ref").distinct()

pacientes_target.count()

36429

In [20]:

features = features.join(pacientes_target, on=["prontuario", "date_ref"], how="inner")
features.count()

9611

In [21]:
features.printSchema()

root
 |-- prontuario: double (nullable = true)
 |-- date_ref: timestamp (nullable = true)
 |-- qtd_internacao_3_0_m: long (nullable = true)
 |-- qtd_internacao_6_3_m: long (nullable = true)
 |-- qtd_internacao_9_6_m: long (nullable = true)
 |-- qtd_internacao_12_9_m: long (nullable = true)
 |-- a_qtd_media_dias_internado: double (nullable = true)
 |-- b_qtd_media_dias_internado: double (nullable = true)
 |-- c_qtd_media_dias_internado: double (nullable = true)
 |-- d_qtd_media_dias_internado: double (nullable = true)
 |-- e_qtd_media_dias_internado: double (nullable = true)
 |-- f_qtd_media_dias_internado: double (nullable = true)
 |-- g_qtd_media_dias_internado: double (nullable = true)
 |-- h_qtd_media_dias_internado: double (nullable = true)
 |-- i_qtd_media_dias_internado: double (nullable = true)
 |-- j_qtd_media_dias_internado: double (nullable = true)
 |-- k_qtd_media_dias_internado: double (nullable = true)
 |-- l_qtd_media_dias_internado: double (nullable = true)
 |-- m_qtd_me

In [22]:
features.toPandas().to_parquet("C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/features_internacao.parquet")

In [23]:
# total rows (avoid recomputing repeatedly) 
total_rows = features.count()

# count nulls and compute percentage 
null_stats = features.select([ F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in features.columns ]) 

# add percentages 
null_stats_percent = null_stats.select([ F.col(c).alias(c + "_nulls") for c in features.columns ] + [ (F.col(c) / total_rows * 100).alias(c + "_pct") for c in features.columns ]) 

null_stats_percent.show()

+----------------+--------------+--------------------------+--------------------------+--------------------------+---------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+-------------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------------+----------------------+------------